## Summary and Next Steps
- Re-run the notebook with your dataset by updating `DATA_PATH`.
- Adjust missing-value strategies for domain knowledge.
- Consider domain-specific outlier rules instead of blanket removal.
- Save cleaned data: `df.to_csv('cleaned_data.csv', index=False)`

# Save cleaned dataset
try:
    df.to_csv('cleaned_data.csv', index=False)
    print('Saved cleaned_data.csv')
except Exception as e:
    print('Could not save file:', e)


In [ ]:
# Optional: small Streamlit app for interactive dashboard
# Save this as streamlit_app.py to run with `streamlit run streamlit_app.py`

streamlit_code = '''
import streamlit as st
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DATA_PATH = 'data.csv'  # replace
try:
    df = pd.read_csv(DATA_PATH)
except Exception:
    df = pd.read_excel(DATA_PATH)

st.title('Data Cleaning Dashboard')
st.write('Shape:', df.shape)

if st.checkbox('Show head'):
    st.dataframe(df.head())

num_cols = df.select_dtypes(include=['number']).columns.tolist()
if num_cols:
    col = st.selectbox('Numeric column', num_cols)
    fig, ax = plt.subplots()
    sns.histplot(df[col].dropna(), kde=True, ax=ax)
    st.pyplot(fig)
'''

print('Streamlit snippet generated. Save to streamlit_app.py to run interactively.')


In [ ]:
## 7. Build Visual Summary Dashboard
# Create a compact dashboard using matplotlib subplots
fig = plt.figure(constrained_layout=True, figsize=(14,10))
spec = fig.add_gridspec(ncols=2, nrows=3)
ax1 = fig.add_subplot(spec[0,0])
ax2 = fig.add_subplot(spec[0,1])
ax3 = fig.add_subplot(spec[1,0])
ax4 = fig.add_subplot(spec[1,1])
ax5 = fig.add_subplot(spec[2,:])

# Top histogram for first numeric column
if len(numeric) > 0:
    sns.histplot(df[numeric[0]].dropna(), kde=True, ax=ax1)
    ax1.set_title(numeric[0])

# Boxplot for second numeric
if len(numeric) > 1:
    sns.boxplot(x=df[numeric[1]], ax=ax2)
    ax2.set_title(numeric[1])

# Correlation heatmap small
if len(numeric) > 1:
    sns.heatmap(df[numeric].corr(), cmap='coolwarm', ax=ax3)
    ax3.set_title('Corr')

# Countplot for first categorical
if len(cat_cols) > 0:
    sns.countplot(y=df[cat_cols[0]], order=df[cat_cols[0]].value_counts().index[:10], ax=ax4)
    ax4.set_title(cat_cols[0])

# Summary text
summary_text = f"Rows: {df.shape[0]}\nColumns: {df.shape[1]}\nDuplicates dropped: {dup_count}\nMissing handled: {df.isnull().sum().sum()}"
ax5.axis('off')
ax5.text(0,0.5, summary_text, fontsize=12)
plt.show()


In [ ]:
## 6. Analyze Relationships and Trends
# Correlation heatmap for numeric columns
if len(numeric) > 1:
    corr = df[numeric].corr()
    plt.figure(figsize=(10,8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
    plt.title('Correlation Heatmap')
    plt.show()

# Example scatter plot of top correlated pair
if len(numeric) > 1:
    # find pair with highest absolute correlation (excluding self)
    corr_abs = corr.abs()
    np.fill_diagonal(corr_abs.values, 0)
    max_idx = np.unravel_index(corr_abs.values.argmax(), corr_abs.shape)
    x_col = corr_abs.columns[max_idx[0]]
    y_col = corr_abs.columns[max_idx[1]]
    print('Top pair:', x_col, 'vs', y_col)
    sns.scatterplot(data=df, x=x_col, y=y_col)
    plt.show()


In [ ]:
## 5. Explore Data Distributions
import matplotlib.pyplot as plt

numeric = df.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric) == 0:
    print('No numeric columns to plot')
else:
    fig, axes = plt.subplots(min(4, len(numeric)), 2, figsize=(12, 4 * min(4, len(numeric))))
    axes = axes.flatten() if len(numeric) > 1 else [axes]
    for i, col in enumerate(numeric[:8]):
        sns.histplot(df[col].dropna(), kde=True, ax=axes[2*i])
        axes[2*i].set_title(f'Histogram: {col}')
        sns.boxplot(x=df[col], ax=axes[2*i+1])
        axes[2*i+1].set_title(f'Boxplot: {col}')
    plt.tight_layout()
    plt.show()


In [ ]:
## 4. Remove or Resolve Duplicates
# Detect duplicates
dup_count = df.duplicated().sum()
print('Total duplicate rows:', dup_count)

# Inspect duplicates (first 5)
if dup_count > 0:
    display(df[df.duplicated(keep=False)].head())

# Drop exact duplicate rows
df = df.drop_duplicates()
print('Shape after dropping duplicates:', df.shape)


In [ ]:
# Example treatments: remove or cap
# To remove rows with outliers in a given column using IQR

def remove_outliers_iqr(df, col, k=1.5):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return df[(df[col] >= lower) & (df[col] <= upper)]

# To cap (winsorize) values
from scipy.stats.mstats import winsorize

def cap_outliers(series, limits=(0.01, 0.01)):
    return winsorize(series, limits=limits)

# Apply removal example for a chosen numeric column
if len(num_cols) > 0:
    col_example = num_cols[0]
    print('Removing outliers in', col_example)
    df = remove_outliers_iqr(df, col_example)
    print('New shape after outlier removal:', df.shape)


In [ ]:
## 3. Detect and Treat Outliers

# IQR-based outlier detection
def iqr_outliers(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return series[(series < lower) | (series > upper)]

# Z-score based detection
def zscore_outliers(series, z_thresh=3.0):
    z = np.abs(stats.zscore(series.dropna()))
    return series.dropna()[z > z_thresh]

# Apply to numeric columns and show counts
for col in num_cols:
    out_iqr = iqr_outliers(df[col])
    out_z = zscore_outliers(df[col])
    if len(out_iqr) > 0 or len(out_z) > 0:
        print(f"{col}: IQR outliers={len(out_iqr)}, Z outliers={len(out_z)}")


In [ ]:
# Strategies: drop rows with >50% missing, impute numeric median, categorical mode
THRESHOLD = 0.5
rows_before = df.shape[0]
# Drop columns with too many missing values
col_drop = df.columns[df.isnull().mean() > THRESHOLD].tolist()
df = df.drop(columns=col_drop)
print('Dropped columns with >50% missing:', col_drop)

# Drop rows if many key fields are missing (example)
# df = df.dropna(subset=['important_col1','important_col2'])

# Impute numeric and categorical
if len(num_cols) > 0:
    df = impute_numeric_median(df, num_cols)
if len(cat_cols) > 0:
    df = impute_categorical_mode(df, cat_cols)

print('Rows before:', rows_before, 'after:', df.shape[0])
print('Remaining missing per column:')
print(df.isnull().sum())

In [ ]:
## 2. Handle Missing Values

# Helper functions for imputation
from sklearn.impute import SimpleImputer

def impute_numeric_median(df, cols):
    imp = SimpleImputer(strategy='median')
    df[cols] = imp.fit_transform(df[cols])
    return df

def impute_categorical_mode(df, cols):
    imp = SimpleImputer(strategy='most_frequent')
    df[cols] = imp.fit_transform(df[cols])
    return df

# Example: decide strategy automatically
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

print('Numeric cols:', num_cols)
print('Categorical cols:', cat_cols)


In [ ]:
# Basic info and missing counts
print(df.info())
print('\nMissing values per column:')
print(df.isnull().sum())
print('\nSummary statistics:')
print(df.describe(include='all'))

In [ ]:
## 1. Load and Inspect Data
# Update the path below to your CSV or Excel file
DATA_PATH = 'data.csv'  # <- replace with your path

# Try CSV, fallback to Excel
try:
    df = pd.read_csv(DATA_PATH)
except Exception:
    df = pd.read_excel(DATA_PATH)

print('Shape:', df.shape)
df.head()


In [ ]:
## 1. Load libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats

# Display settings
pd.options.display.max_columns = 50
sns.set(style='whitegrid')

# Data Cleaning and Dashboard
This notebook loads a dataset, handles missing values, detects and treats outliers, removes duplicates, and creates visual reports and a dashboard using pandas, matplotlib, and seaborn. Replace `data.csv` with your file path.